# Importy

In [28]:
# Standardowa biblioteka
import time
from pathlib import Path
from dataclasses import dataclass, asdict, replace

# Dane i wykresy
import numpy as np
import matplotlib.pyplot as plt

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

# Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.initializers import HeNormal, GlorotUniform
from tensorflow.keras.regularizers import L2
from tensorflow.keras.optimizers import SGD, Adam, AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras import losses
from tensorflow.keras import metrics as keras_metrics
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed

# Ładowanie danych i usunięcie niepotrzebnych kolumn

In [29]:
DATA_PATH = Path("../data/high_diamond_ranked_10min.csv")

TARGET = "blueWins"
DROP_COLUMNS = [
    "gameId",
    "redFirstBlood", "redKills", "redDeaths",
    "blueGoldDiff", "blueExperienceDiff", "redGoldDiff", "redExperienceDiff",
    "blueEliteMonsters", "redEliteMonsters",
    "blueCSPerMin", "redCSPerMin", "blueGoldPerMin", "redGoldPerMin",
]

def load_data(path):
    names = np.loadtxt(path, delimiter=",", max_rows=1, dtype=str)

    numbers = np.loadtxt(path, delimiter=",", skiprows=1, dtype=float)

    mask = ~np.isin(names, DROP_COLUMNS + [TARGET])
    target_mask = names == TARGET

    X = numbers[:,  mask]
    y = numbers[:, target_mask].ravel() 
    feature_names = names[mask]


    return X, y, feature_names


X, y, feature_names = load_data(DATA_PATH)


print(f"X shape: {X.shape}")
print(f"target check: {np.unique(y)}")
print(f"feature names: {feature_names}")

X shape: (9879, 25)
target check: [0. 1.]
feature names: ['blueWardsPlaced' 'blueWardsDestroyed' 'blueFirstBlood' 'blueKills'
 'blueDeaths' 'blueAssists' 'blueDragons' 'blueHeralds'
 'blueTowersDestroyed' 'blueTotalGold' 'blueAvgLevel'
 'blueTotalExperience' 'blueTotalMinionsKilled'
 'blueTotalJungleMinionsKilled' 'redWardsPlaced' 'redWardsDestroyed'
 'redAssists' 'redDragons' 'redHeralds' 'redTowersDestroyed'
 'redTotalGold' 'redAvgLevel' 'redTotalExperience' 'redTotalMinionsKilled'
 'redTotalJungleMinionsKilled']


# Podział danych oraz standaryzacja

In [30]:
X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.3, random_state=2026, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_hold, y_hold, test_size=0.5, random_state=2026, stratify=y_hold)

y_train = y_train.astype("float32").reshape(-1, 1)
y_val = y_val.astype("float32").reshape(-1, 1)
y_test = y_test.astype("float32").reshape(-1, 1)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype("float32")
X_val_s = scaler.transform(X_val).astype("float32")
X_test_s = scaler.transform(X_test).astype("float32")

print("Training / Validation / Test:", len(X_train), len(X_val), len(X_test))
print("Bincount (0, 1):",np.bincount(y_train.ravel().astype("int32")), np.bincount(y_val.ravel().astype("int32")), np.bincount(y_test.ravel().astype("int32")))

Training / Validation / Test: 6915 1482 1482
Bincount (0, 1): [3464 3451] [743 739] [742 740]


# Konfiguracja

In [31]:
from dataclasses import dataclass, asdict, replace

@dataclass
class ExperimentConfig:
    name: str
    hidden_layers: int =2
    units: int = 32
    dropout: float = 0.0
    l2: float = 0.0
    optimizer: str = 'adam'
    learning_rate: float = 1e-3
    momentum: float = 0.0
    weight_decay: float = 0.0
    clipnorm: float | None = None
    batch_size: int = 32
    epochs: int = 100
    early_stopping: bool = False
    reduce_lr: bool = False
    seed: int = 2026

# Funkcje pomocnicze

In [ ]:
def build_model(config, n_features):
    model = Sequential()

    model.add(Input(shape=(n_features,)))

    for i in range(config.hidden_layers):
        model.add(Dense(config.units, activation='relu',
                        kernel_initializer=HeNormal(seed=config.seed + i),
                        kernel_regularizer=L2(config.l2) if config.l2 > 0.0 else None
                        ))
        if config.dropout > 0.0:
            model.add(Dropout(config.dropout))
    
    model.add(Dense(1, activation='sigmoid'))

    return model

def build_optimizer(config):
    if config.weight_decay > 0.0 and config.optimizer != "adamw":
        raise ValueError("weight_decay is only applicable for AdamW optimizer. Please set optimizer to 'adamw' or set weight_decay to 0.0.")


    if config.optimizer == "sgd":
        return SGD(learning_rate=config.learning_rate, momentum=config.momentum, clipnorm=config.clipnorm)
    elif config.optimizer == "adam":
        return Adam(learning_rate=config.learning_rate, clipnorm=config.clipnorm)
    elif config.optimizer == "adamw":
        optimizer = AdamW(learning_rate=config.learning_rate, weight_decay=config.weight_decay, clipnorm=config.clipnorm)
        optimizer.exclude_from_weight_decay(var_names=["bias"])

        return optimizer
    else:
        raise ValueError(f"Unknown optimizer: {config.optimizer}")


Epoch 1/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 986us/step - accuracy: 0.6895 - loss: 0.5819 - val_accuracy: 0.7139 - val_loss: 0.5527
Epoch 2/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 0s 800us/step - accuracy: 0.7235 - loss: 0.5433 - val_accuracy: 0.7173 - val_loss: 0.5490
Epoch 3/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 0s 760us/step - accuracy: 0.7300 - loss: 0.5342 - val_accuracy: 0.7227 - val_loss: 0.5423
Epoch 4/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - accuracy: 0.7383 - loss: 0.5285 - val_accuracy: 0.7240 - val_loss: 0.5412
Epoch 5/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 0s 781us/step - accuracy: 0.7403 - loss: 0.5248 - val_accuracy: 0.7240 - val_loss: 0.5401
Epoch 6/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 0s 797us/step - accuracy: 0.7404 - loss: 0.5213 - val_accuracy: 0.7254 - val_loss: 0.5450
Epoch 7/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 0s 732us/step - accuracy: 0.7430 - loss: 0.5188 - val_accuracy: 0.7247 - val_loss: 0.5420
Epoch 8/10
217/217 ━━━━━━━━━━━━━━━━━━━━ 0s 745us/step - accuracy: 0.7422 - loss: 0.5157 - 